# 05 — Phase B: in-domain features, final training, export

1. Reloads the **base bnb4** model with the live Robert+VARM3 prompt prefix and
   teacher-forces the generated replies (their own codes + text tokens — no
   alignment guessing here) → live-distribution features.
2. Fine-tunes `head_phaseA.pt` on Phase A + Phase B (or trains from scratch).
3. Grid-searches the hysteresis smoother (switches/min target) on the in-domain
   test set.
4. Exports `helium_emotion_head_final.pt`, `config.json`, `eval_report.json`.

Restart the kernel first if 04 just ran (frees the bf16 weights).


In [ ]:
from pathlib import Path
import os, sys, json, time
import numpy as np
import torch

sys.path.insert(0, os.getcwd())
import eh_common as eh

P = eh.get_paths()   # override: P = eh.get_paths("/your/root") or set EMOTION_HEAD_ROOT
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16
print("ROOT:", P.ROOT, "| device:", DEVICE)
for k in ("MOSHI_BNB4", "MIMI_FILE", "SPM_FILE", "VOICE_PT"):
    p = Path(getattr(P, k))
    print(f"  {k}: {'OK' if p.exists() else 'MISSING'}  {p}")


In [ ]:
import collections
import pandas as pd
FINETUNE = True            # False -> train from scratch on A+B
LR = 1e-4
EPOCHS = 8
BATCH_SIZE = 16
TARGET_SWITCHES_PER_MIN = 0.5


In [ ]:
sp = eh.load_sp(P)
mimi = eh.load_mimi(P, DEVICE)
lm = eh.load_lm_bnb4(P, DEVICE, DTYPE)
lm_gen = eh.make_lm_gen(lm, DEVICE)
eh.install_hidden_step(lm_gen)
assert P.TEXT_PROMPT_FILE.exists() and P.VOICE_PT.exists()
prompt_state = eh.apply_prompts_and_snapshot(
    lm_gen, mimi, sp, str(P.VOICE_PT),
    P.TEXT_PROMPT_FILE.read_text(encoding="utf-8", errors="replace"))
print("live prompt prefix ready; cuda mem:",
      f"{torch.cuda.memory_allocated()/1e9:.1f} GiB")


In [ ]:
from tqdm.auto import tqdm
man = pd.read_parquet(P.DATA / "manifest.parquet")
user_of = dict(zip(man.clip_id, man.user_id))
holdout_users = set(man.loc[man.split == "holdout", "user_id"].unique())

writer = eh.ShardWriter(P.FEAT_B, "phaseB", clips_per_shard=128)
gen_recs = list(eh.iter_shard_records(P.GEN, "gen"))
print("generated samples:", len(gen_recs), "| already extracted:", len(writer.completed))
n_test = max(1, len(gen_recs) // 10)
test_ids = set(r["clip_id"] for r in gen_recs[-n_test:])   # last 10% = in-domain test

log = eh.ProgressLogger(P.FEAT_B / "extract_phaseB.log", echo_every=25)
todo = [r for r in gen_recs if r["clip_id"] not in writer.completed]
total_target = len(writer.completed) + len(todo)
log.log(f"=== phaseB extraction START | completed={len(writer.completed)} "
        f"todo={len(todo)} target={total_target}", force=True)
t0 = time.time()
failed, times = 0, []
for r in tqdm(todo, total=len(todo)):
    ct0 = time.time()
    try:
        eh.restore_streaming_state(lm_gen, mimi, prompt_state)
        feats = eh.extract_from_codes(lm_gen, mimi, r["user_codes"].long(), r["codes"].long(),
                                      r["text_tokens"].long().numpy(), DEVICE)
        T = feats["hidden"].shape[0]
        rms = r["rms"].float().numpy()
        mask = np.zeros(T, dtype=bool)
        for t in range(int(r["n_user_frames"]), T):
            mask[t] = rms[t] > 0.006
        uid = r.get("user_id") or user_of.get(r["src_clip"], "")
        if uid in holdout_users:
            split = "holdout"
        elif r["clip_id"] in test_ids:
            split = "test_indomain"
        else:
            split = "train"
        feats["pros"] = torch.from_numpy(eh.prosody_features(r["pcm"].float().numpy()))
        writer.add(dict(clip_id=r["clip_id"], **feats,
                        agent_mask=torch.from_numpy(mask), label=int(r["label"]),
                        user_emotion=-1, user_id=uid, split=split,
                        text_stream=r["text_tokens"]))
    except Exception as e:
        failed += 1
        log.log(f"ERROR sample={r['clip_id']}: {type(e).__name__}: {e} "
                f"| skipped (resume-safe)", force=True)
        continue
    dt = time.time() - ct0
    times.append(dt)
    completed = len(writer.completed)
    remaining = max(0, total_target - completed)
    avg = sum(times) / len(times)
    log.log(f"sample {completed}/{total_target} | +{dt:.2f}s avg={avg:.2f}s "
            f"| left={remaining} ETA={avg * remaining / 3600:.2f}h "
            f"| failed={failed} | {r['clip_id']}")
writer.close()
log.log(f"=== phaseB extraction END | completed={len(writer.completed)}/{total_target} "
        f"failed={failed} | elapsed={(time.time() - t0) / 3600:.2f}h", force=True)
print("phaseB extracted:", len(writer.completed), "| failed:", failed, "| log:", log.path)
del lm, lm_gen, mimi
eh.free_gpu()


In [ ]:
recsA = list(eh.iter_shard_records(P.FEAT_A, "phaseA"))
recsB = list(eh.iter_shard_records(P.FEAT_B, "phaseB"))
tr = [r for r in recsA if r["split"] == "train"] + [r for r in recsB if r["split"] == "train"]
va = [r for r in recsA if r["split"] == "val"] or [r for r in recsB if r["split"] == "test_indomain"]
ho = [r for r in recsA if r["split"] == "holdout"] + [r for r in recsB if r["split"] == "holdout"]
te = [r for r in recsB if r["split"] == "test_indomain"]
nA = sum(1 for r in tr if not str(r["clip_id"]).startswith("gen_"))
print(f"train {len(tr)} (A {nA} + B {len(tr)-nA}) | val {len(va)} | holdout {len(ho)} | testB {len(te)}")

class SeqDataset(torch.utils.data.Dataset):
    def __init__(self, recs):
        self.recs = recs
    def __len__(self):
        return len(self.recs)
    def __getitem__(self, i):
        r = self.recs[i]
        return (r["hidden"], r["mimi_lat"], r["pros"], r["agent_mask"], int(r["label"]))

def collate(batch):
    T = max(b[0].shape[0] for b in batch)
    B = len(batch)
    H = torch.zeros(B, T, eh.HELIUM_DIM)
    M = torch.zeros(B, T, eh.MIMI_LATENT_DIM)
    PR = torch.zeros(B, T, eh.PROSODY_DIM)
    MK = torch.zeros(B, T, dtype=torch.bool)
    LB = torch.zeros(B, dtype=torch.long)
    for i, (h, m, p, mk, lb) in enumerate(batch):
        t = h.shape[0]
        H[i, :t] = h.float()
        M[i, :t] = m.float()
        PR[i, :t] = p.float()
        MK[i, :t] = mk
        LB[i] = lb
    return H, M, PR, MK, LB

g = torch.Generator().manual_seed(eh.SEED)
train_loader = torch.utils.data.DataLoader(
    SeqDataset(tr), batch_size=BATCH_SIZE, shuffle=True,
    collate_fn=collate, generator=g, drop_last=True)
val_loader = torch.utils.data.DataLoader(
    SeqDataset(va), batch_size=BATCH_SIZE, collate_fn=collate)


In [ ]:
if FINETUNE and (P.MODELS / "head_phaseA.pt").exists():
    model = eh.load_head(P.MODELS / "head_phaseA.pt", DEVICE)
    print("fine-tuning from head_phaseA.pt")
else:
    model = eh.EmotionFusionHead()
    print("training from scratch")
best_f1, history = eh.train_head(model, train_loader, val_loader, device=DEVICE,
                                 epochs=EPOCHS, lr=LR, patience=3)
print("best val macro-F1:", round(best_f1, 4))


In [ ]:
te_loader = torch.utils.data.DataLoader(SeqDataset(te or ho), batch_size=BATCH_SIZE,
                                        collate_fn=collate)
logits, labels, masks = eh.evaluate_head(model, te_loader, DEVICE)
f1_final = eh.macro_f1_from_logits(logits, labels, masks)
print(f"in-domain test macro-F1: {f1_final:.4f}")

best_cfg, best_score = None, -1e9
for a in (0.3, 0.4, 0.5):
    for mrg in (0.10, 0.15, 0.20):
        for dw in (1.0, 1.5, 2.0):
            sw, correct, total = [], 0, 0
            for i in range(logits.shape[0]):
                m = masks[i]
                if not m.any():
                    continue
                sm = eh.HysteresisSmoother(ema_alpha=a, enter_margin=mrg, min_dwell_s=dw)
                votes = collections.Counter()
                for t in range(logits.shape[1]):
                    if m[t]:
                        eid, _, _ = sm.update(torch.softmax(logits[i, t].float(), -1).numpy())
                        votes[eid] += 1
                sw.append(sm.switches_per_min())
                correct += int(votes.most_common(1)[0][0] == int(labels[i]))
                total += 1
            spm = float(np.mean(sw))
            acc = correct / max(1, total)
            score = acc - max(0.0, spm - TARGET_SWITCHES_PER_MIN)
            if score > best_score:
                best_score = score
                best_cfg = dict(ema_alpha=a, enter_margin=mrg, min_dwell_s=dw,
                                switches_per_min=spm, turn_acc=acc)
print("chosen smoother config:", {k: round(v, 4) if isinstance(v, float) else v
                                  for k, v in best_cfg.items()})


In [ ]:
eh.save_head(model, P.MODELS / "helium_emotion_head_final.pt",
             extra=dict(val_macro_f1=best_f1, indomain_test_macro_f1=f1_final,
                        history=history))
config = dict(head=dict(model.cfg), emotions=eh.EMOTIONS, smoother=best_cfg,
              temperature=1.0, frame_hz=eh.FRAME_HZ, helium_dim=eh.HELIUM_DIM,
              mimi_latent_dim=eh.MIMI_LATENT_DIM, prosody_dim=eh.PROSODY_DIM)
(P.MODELS / "config.json").write_text(json.dumps(config, indent=2))
report = dict(phaseB_samples=len(recsB), val_macro_f1=best_f1,
              indomain_test_macro_f1=f1_final, smoother=best_cfg)
(P.REPORTS / "eval_report.json").write_text(json.dumps(report, indent=2))
print("EXPORTED:", P.MODELS / "helium_emotion_head_final.pt")
print("         ", P.MODELS / "config.json")
print("→ continue with 06_deploy_and_live_replay.ipynb")
